# Web3 Python 100本ノック：第1章
## §1-3：残高の一括処理取得

§1-1、§1-2では単一のアドレスの残高を取得しました。
しかし、実際のWeb3データ分析やシステム運用では、「自社の複数ウォレットの残高を監視する」「大口投資家（クジラ）の複数アドレスをトラッキングする」といった**自動化・一括処理**が求められます。

この問題では、Pythonの得意分野である「リスト」と「forループ」を活用し、複数のアドレスのETH残高を連続で取得するスクリプトを作成します。

このノートでは、Pythonからスマートコントラクトの関数を呼び出し、特定のトークン残高を取得する方法を学びます。


> **注意**: 各セルを順番に実行してください。セルの実行には `Shift + Enter` を押します。

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Tsukumo-999/web3-knock100-first-half/blob/master/s1-3_get_by_list.ipynb)

## 準備
環境がリセットされている場合は、以下のセルでライブラリをインストールしてください。

In [5]:
# !pip install web3==7.16.0

## 実装のポイント：ループ処理とレートリミット対策

パブリックRPCに対して、プログラムで一気に大量の通信（リクエスト）を送ると、攻撃とみなされて通信を遮断される（レートリミットに引っかかる）可能性があります。
そのため、`time.sleep()` を使って**意図的に少し待機時間を設ける**のが、Web3スクリプトを安定して動かすための重要なお作法です。

### 1. 接続設定

In [6]:
from web3 import Web3
import time  # 待機時間を設定するための標準ライブラリ


RPC_URL = "https://eth.drpc.org"
headers = {'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36'}
w3 = Web3(Web3.HTTPProvider(RPC_URL, request_kwargs={'headers': headers, 'timeout': 5}))

try:
    latest_block = w3.eth.block_number
    print(f"接続成功！ (最新ブロック: {latest_block})")
except Exception as e:
    print(f"接続失敗: {e}")

接続成功！ (最新ブロック: 25658919)


### 2. 監視したいアドレスのリストを作成
今回は有名なアドレスを3つ用意

In [7]:
# 2. 監視したいアドレスのリストを作成（今回は有名なアドレスを3つ用意）
target_addresses = [
    "0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045", # Vitalik Buterin氏
    "0x28C6c06298d514Db089934071355E5743bf21d60", # Binance (取引所)
    "0x00000000219ab540356cBB839Cbe05303d7705Fa"  # Eth2 Deposit Contract (ネットワーク全体のステーキング資金)
]

print("ウォレット一括照会セットアップ完了")

ウォレット一括照会セットアップ完了


### 3. ループ処理で順番に残高を取得
リストにしたアドレスを順番に取得

In [8]:
# 3. ループ処理で順番に残高を取得
total_eth_monitored = 0

for addr in target_addresses:
    try:
        # Web3のお作法：アドレスの大文字小文字を正規化（チェックサム）して安全にする
        checksum_addr = w3.to_checksum_address(addr)
        
        # 残高取得と変換
        bal_wei = w3.eth.get_balance(checksum_addr)
        bal_eth = w3.from_wei(bal_wei, "ether")
        
        # 合計値に加算
        total_eth_monitored += bal_eth
        
        print(f"Address: {checksum_addr}")
        print(f"Balance: {bal_eth:,.2f} ETH")
        print("-" * 40)
        
        # パブリックRPCへの負荷軽減のため、0.2秒待機
        time.sleep(0.2)
        
    except Exception as e:
        print(f"{addr} の取得に失敗しました: {e}")

# 4. 最後に合計値を表示
print(f"監視アドレスの合計残高: {total_eth_monitored:,.2f} ETH")
print("=" * 40)

Address: 0xd8dA6BF26964aF9D7eEd9e03E53415D37aA96045
Balance: 6.63 ETH
----------------------------------------
Address: 0x28C6c06298d514Db089934071355E5743bf21d60
Balance: 132,971.64 ETH
----------------------------------------
Address: 0x00000000219ab540356cBB839Cbe05303d7705Fa
Balance: 89,002,739.63 ETH
----------------------------------------
監視アドレスの合計残高: 89,135,717.90 ETH
